# 第10回　アンサンブル学習
***
> **前提**: 第8回 Pipeline と異なり，本回は**手動前処理**で復習しながらアンサンブル学習を学びます。問題1では第8回と同様，**train/test 分割のあと訓練データだけの中央値で `Age` を補完**します（テスト情報を混ぜない）。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその手法を選ぶのか**」「**パラメータを変えると過学習や精度がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. ランダムフォレスト
2. 勾配ブースティング
3. XGBoost
4. 特徴量重要度

---

## この回で学ぶこと

### アンサンブル学習とは

「複数の弱いモデルを組み合わせて強いモデルを作る」手法だ。選挙の多数決に例えると分かりやすい：1人の専門家より100人の多数決の方が信頼できる，という考え方と同じだ。

アンサンブル学習には大きく2つの流れがある：

```
【バギング（Bagging）】
  ランダムに異なる訓練データ → 複数の木を並列学習 → 多数決
  └ RandomForest がこれ

【ブースティング（Boosting）】
  前のモデルの間違いを次のモデルが補正 → 逐次的に学習
  └ GradientBoosting, XGBoost がこれ
```

### ランダムフォレスト（Random Forest）

- 複数の決定木をランダムなデータ・特徴量のサブセットで学習
- **過学習しにくい**：各木が異なるパターンを学習し，多数決で平均的な予測をする
- `n_estimators`：木の本数（多いほど安定するが，計算時間も増える）
- **並列化できる**ため，大規模データでも速い

### 勾配ブースティング（Gradient Boosting）

- 前の木の**残差（誤差）**を次の木が学習する逐次プロセス
- 精度は高いがチューニングが難しく，過学習しやすい
- `n_estimators`（木の本数）と `learning_rate`（学習率）のバランスが重要

### XGBoost

勾配ブースティングを大幅に改良したライブラリ：
- **2次の損失関数近似**で収束が速い
- **L1/L2 正則化**を内蔵（過学習を防ぐ）
- 欠損値を自動処理
- 並列計算・GPU 対応

Kaggle などのデータコンペで長年トップの手法として使われてきた。実務でも最も多用されるアルゴリズムの一つだ。

### 特徴量重要度（Feature Importance）の注意点

ランダムフォレストの `feature_importances_` は各特徴量が分岐にどれだけ貢献したかを示す。ただし：
- **相関する特徴量があると重要度が分散**する（連動して動く変数は各々の重要度が小さく見える）
- 重要度が高い = 目的変数の「原因」とは限らない（**相関≠因果**）
- 卒業研究で「特徴量重要度が高い変数がXXの原因だ」という解釈は慎重に行うこと

In [ ]:
%pip install -q xgboost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import pandas as pd

TITANIC_URL = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data/titanic/titanic.csv"

def load_titanic() -> pd.DataFrame:
    return pd.read_csv(TITANIC_URL)


# === ★ここを変えてモデルを差し替える★（問題1・3で使用）===
MODEL_REGISTRY = {
    "dt": lambda rs=0: DecisionTreeClassifier(random_state=rs),
    "rf": lambda rs=0: RandomForestClassifier(n_estimators=100, random_state=rs),
    "gb": lambda rs=0: GradientBoostingClassifier(random_state=rs),
    "xgb": lambda rs=0: XGBClassifier(random_state=rs, eval_metric="logloss"),
}
MODEL_LABELS = {
    "dt": "DecisionTree (単一木)",
    "rf": "RandomForest (バギング)",
    "gb": "GradientBoosting (ブースティング)",
    "xgb": "XGBoost (ブースティング系)",
}


def prep_titanic_xy():
    """train/test 分割後、訓練データの中央値だけで Age を補完（第8回 Pipeline と同じ考え方）。"""
    use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
    df = load_titanic()[use_cols].copy()
    X = df.drop(columns="Survived")
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0, stratify=y
    )
    age_med = X_train["Age"].median()
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train["Age"] = X_train["Age"].fillna(age_med)
    X_test["Age"] = X_test["Age"].fillna(age_med)
    X_train = pd.get_dummies(X_train, columns=["Sex"], drop_first=True)
    X_test = pd.get_dummies(X_test, columns=["Sex"], drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test, y_train, y_test


def print_train_test_scores(name, model, X_train, X_test, y_train, y_test):
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:32s} train={tr:.4f}  test={te:.4f}  差={tr - te:+.4f}")
    return tr, te


## 問題1　手法選択：単一木 / バギング / ブースティング　【骨格+選択】
***

### `random_state` を設定する理由

`random_state=0` を指定することで，毎回同じ乱数が使われ，**結果が再現可能**になる。研究論文では再現性が必須のため，乱数シードの管理は重要なルールだ。

### `n_estimators=100` の選び方

木の本数が多いほどモデルは安定するが，計算時間が増える。一般的に：
- 100〜500: 多くの場合で十分
- 1000以上: 計算コストが高く，精度向上が頭打ちになりやすい

**実用的なアドバイス**: まず100から始め，精度に問題があれば徐々に増やす。

### バイアスとバリアンス（手法選択の核心）

- **単一の決定木**：木を深く育てると訓練データを「暗記」できる → **バイアス低・バリアンス高**（＝過学習しやすい）。訓練精度は高いがテスト精度が落ちる。
- **バギング（RandomForest）**：少しずつ異なる多数の木を作り**平均（多数決）**する → バリアンス（ばらつき）を下げて過学習を抑える。
- **ブースティング（GradientBoosting）**：前の木の誤りを次の木が補正していく → 主に**バイアス**を下げて精度を上げるが、やりすぎると過学習する。

### 課題

下のコードセルは前処理（分割後に訓練データの中央値で `Age` 補完、`Sex` はダミー変数化）と `MODEL_REGISTRY` によるモデル差し替えまで用意してあります。**核心（各モデルの学習）はあなたが書きます**（`# ★あなたが書く★`）。`model_keys_p1` を変えれば DT / RF / GB / XGB を比較できます。まずは **dt・rf・gb** の **train 正解率・test 正解率・その差**を観察してください。

観察したうえで、次の **設計判断** に答えてください。

> **設計判断1**: 「過学習を抑えつつ、まず安定して良い精度を出したい」とき、次のどれを使うのが適切か**1つ選び、理由**を解答用コードセルに書いてください。**バイアス・バリアンスの観点**で根拠を書くこと。
>
> - **(A) 単一の決定木（`DecisionTreeClassifier`、深さ無制限）**
> - **(B) RandomForest（バギング）**
> - **(C) GradientBoosting（ブースティング）**
> - **(D) XGBoost（ブースティング系）**
> - **(E) 決定木を `max_depth` で浅く制限する（アンサンブルしない）**
> - **(F) アンサンブルせず線形モデル（`LogisticRegression`）を使う**
>
> ヒント：出力の「train と test の差」を見てください。差が大きい手法は過学習しています。どの手法が train だけ極端に高くなっているか、どの手法が train と test がバランスしているかを根拠にしてください。「正解は1つ」ではなく、**根拠が筋の通った選択**であることが重要です。

In [ ]:
# === 完成済みコード：そのまま実行して、train / test 正解率を比較してください ===
X_train, X_test, y_train, y_test = prep_titanic_xy()
X = X_train  # 問題4 の feature_names 用

# === ★ここを変えて比較するモデルを選ぶ★（キーは MODEL_REGISTRY と同じ）===
model_keys_p1 = ["dt", "rf", "gb"]

rf = None  # 問題4 で feature_importances_ に使用
for key in model_keys_p1:
    model = MODEL_REGISTRY[key]()
    if key == "rf":
        rf = model
    # ★あなたが書く★：model を訓練データ（X_train, y_train）で学習する（1行）
    #   ヒント: model.fit(説明変数, 目的変数)
    ___
    print_train_test_scores(MODEL_LABELS[key], model, X_train, X_test, y_train, y_test)


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
import pandas as pd


# (1-a) 観察（3手法の train / test 正解率を記入）
observation_1_a = pd.DataFrame([
    {'手法': 'DecisionTree（単一木）', 'train_acc': None, 'test_acc': None, 'gap': None},
    {'手法': 'RandomForest（バギング）', 'train_acc': None, 'test_acc': None, 'gap': None},
    {'手法': 'GradientBoosting（ブースティング）', 'train_acc': None, 'test_acc': None, 'gap': None},
])

# (1-b) 設計判断1：過学習を抑えつつ安定して良い精度を出したいとき選んだ手法（A〜F）：(　)
# (A) 単一の決定木（深さ無制限）
# (B) RandomForest（バギング）
# (C) GradientBoosting（ブースティング）
# (D) XGBoost（ブースティング系）
# (E) 決定木を max_depth で浅く制限する（アンサンブルしない）
# (F) 線形モデル（LogisticRegression など）
design1_choice = ""

# (1-c) その理由（バイアス・バリアンスと train/test の差に触れて）
answer_1_c = """
"""

# (1-d) 最も過学習していた（train と test の差が大きい）手法はどれか
answer_1_d = """
"""



## 問題2　ハイパーパラメータと過学習の実験（GradientBoosting）　【実験】
***

### 乳がんデータセットについて

`load_breast_cancer()` は腫瘍の細胞核の形態学的特徴（半径，テクスチャ，周囲長など）30個から「良性/悪性」を判定する有名なベンチマークデータだ。
- サンプル数: 569件
- 特徴量: 30個（すべて数値）
- クラス: 良性（357件）/ 悪性（212件）

医療分野では「悪性を見逃さない（再現率を高める）」ことが特に重要になる（第12回で詳しく扱う）。

### 勾配ブースティングのデフォルトパラメータ

`GradientBoostingClassifier()` のデフォルト値：
- `n_estimators=100`：木の本数
- `learning_rate=0.1`：各ステップの寄与率（小さいほど慎重に学習）
- `max_depth=3`：各木の深さ（浅い木を多数組み合わせる設計）

`learning_rate` を小さくする場合，`n_estimators` を増やすと精度が維持される（トレードオフの関係）。

### 過学習を生むパラメータ

ブースティングは強力だが、設定次第で簡単に過学習する。特に効くのは次の3つ：
- `n_estimators`（木の本数）：多いほど複雑になる
- `max_depth`（各木の深さ）：深いほど1本の木が複雑になる
- `learning_rate`（学習率）：大きいほど一気に学習する（過学習しやすい）

「**訓練精度は高いのにテスト精度が低い**」状態が過学習のサインだ。

### 課題

下のコードセルは実験ループの骨組みを用意してありますが、**核心（GradientBoosting の学習）はあなたが書きます**（`# ★あなたが書く★`）。乳がんデータに対して、`settings` の `(n_estimators, max_depth, learning_rate)` の組合せごとに **train 正解率・test 正解率・その差**を出力します。

`settings` の組合せを **5通り以上**試し（控えめな設定〜過激な設定）、しかも `max_depth` と `learning_rate` の **2つ以上の軸**を動かして、結果を **✍️ 解答用コードセルの実験ログ**に記入してください。

そのうえで考察してください：

> **考察1**: `max_depth` や `learning_rate` を大きくしたとき、train と test の差（過学習の度合い）はどう変わりましたか？ 最も過学習していた組合せはどれでしたか？
>
> **考察2**: `learning_rate` を小さくすると慎重な学習になります。その分テスト精度を保つには `n_estimators`（木の本数）をどうすべきだと思いますか？（ヒント：両者はトレードオフの関係）


In [ ]:
# === 完成済みコード：n_estimators / max_depth / learning_rate を変えて過学習を観察する ===
data = load_breast_cancer()
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=0
)

# === ★ここを変えて実験する★：(n_estimators, max_depth, learning_rate) の組合せ（5通り以上・2軸以上） ===
settings = [
    (100, 3, 0.1),   # デフォルト相当（控えめ）
    (100, 1, 0.1),   # 浅い木
    (100, 5, 0.1),   # 深い木（depth 軸）
    (100, 3, 0.5),   # 学習率大（lr 軸）
    (500, 5, 0.5),   # 過激（過学習しやすい）
    # 例: (50, 3, 0.01) などを追加してよい
]

print(f"{'n_est':>6s} {'depth':>6s} {'lr':>5s} {'train':>8s} {'test':>8s} {'差':>8s}")
print("-" * 44)
for n_est, depth, lr in settings:
    gb = GradientBoostingClassifier(
        n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=0
    )
    # ★あなたが書く★：gb を訓練データ（Xb_train, yb_train）で学習する（1行）
    #   ヒント: gb.fit(説明変数, 目的変数)
    ___
    tr = accuracy_score(yb_train, gb.predict(Xb_train))
    te = accuracy_score(yb_test, gb.predict(Xb_test))
    print(f"{n_est:6d} {depth:6d} {lr:5.2f} {tr:8.4f} {te:8.4f} {tr - te:+8.4f}")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ（5通り以上・max_depth と learning_rate の両軸を動かす）
experiment_log = pd.DataFrame([
    {'row': 1, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'train_acc': None, 'test_acc': None, 'gap': None},
    {'row': 2, 'n_estimators': 100, 'max_depth': 1, 'learning_rate': 0.1, 'train_acc': None, 'test_acc': None, 'gap': None},
    {'row': 3, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'train_acc': None, 'test_acc': None, 'gap': None},
    {'row': 4, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.5, 'train_acc': None, 'test_acc': None, 'gap': None},
    {'row': 5, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.5, 'train_acc': None, 'test_acc': None, 'gap': None},
    {'row': 6, 'n_estimators': None, 'max_depth': None, 'learning_rate': None, 'train_acc': None, 'test_acc': None, 'gap': None},
])

# (2-b) 最も過学習していた（train と test の差が最大の）組合せ
answer_2_b = """
"""

# (2-c) 考察1：max_depth / learning_rate を大きくしたときの過学習の変化
reflection1 = """
"""

# (2-d) 考察2：learning_rate を小さくしたら n_estimators をどうすべきか
reflection2 = """
"""



## 問題3　XGBoost と3モデル比較・使い分けの選択　【骨格+選択】
***

### XGBoost のインストールについて

XGBoost は scikit-learn に含まれない外部ライブラリだ。最初のセルで `%pip install xgboost` を実行している。このように機械学習の実践では様々なライブラリを組み合わせて使う。

### `eval_metric="logloss"` の意味

`logloss`（対数損失）は2値分類の損失関数で，XGBoost が内部で学習の進捗を評価するために使う。警告を抑制するために明示的に指定している。

### 3モデルの使い分けガイド

| モデル | 長所 | 短所 | 向いている場面 |
|---|---|---|---|
| RandomForest | 過学習しにくい，並列計算可能 | 精度は他より劣ることも | データが少ない，外れ値が多い |
| GradientBoosting | 精度が高い | 遅い，チューニング必要 | 精度優先，データが中程度 |
| XGBoost | 精度高い，速い，欠損値対応 | やや複雑 | コンペ，実務，大規模データ |

### 課題

下のコードセルは `MODEL_REGISTRY` と `model_keys_p3` で比較する骨組みを用意してありますが、**核心（各モデルの学習）はあなたが書きます**（`# ★あなたが書く★`）。書いて実行し、同じ乳がんデータでの **train 正解率・test 正解率**を比較してください（初期設定は rf・gb・xgb）。

そのうえで、次の **設計判断** に答えてください。

> **設計判断2（手法の使い分け）**: 次の2つの状況それぞれで、最も向いていると思う手法を下の選択肢から**1つずつ選び、理由**を解答用コードセルに書いてください。
>
> 1. **データにノイズ・外れ値が多く、過学習を絶対に避けたい**とき
> 2. **多少チューニングしてでも最高精度を引き出したい**（Kaggle のコンペなど）とき
>
> - **(A) 単一決定木**
> - **(B) RandomForest（バギング）**
> - **(C) GradientBoosting（ブースティング）**
> - **(D) XGBoost（ブースティング系）**
> - **(E) LogisticRegression（線形・アンサンブルなし）**
> - **(F) スタッキング（複数モデルの予測をさらに別モデルで統合）**
>
> ヒント：ブースティングは「前の誤りを次が補正する」ため、**ノイズや外れ値まで一生懸命学習して過学習しやすい**。バギングは多数の木の平均で**ばらつきに強い**。問題2で見た「過学習のしやすさ」も思い出してください。

> **考察3**: このシンプルなデータでは3モデルの差は小さいはずです。それでも実務で XGBoost が好まれる理由を、`この回で学ぶこと` の表を参考に1つ挙げてください。


In [ ]:
# === 完成済みコード：同じ乳がんデータでモデルを比較する（Xb_* は問題2で作成済み）===
# === ★ここを変えて比較するモデルを選ぶ★ ===
model_keys_p3 = ["rf", "gb", "xgb"]

compare = {}
for key in model_keys_p3:
    model = MODEL_REGISTRY[key]()
    # ★あなたが書く★：model を訓練データ（Xb_train, yb_train）で学習する（1行）
    #   ヒント: model.fit(説明変数, 目的変数)
    ___
    tr = accuracy_score(yb_train, model.predict(Xb_train))
    te = accuracy_score(yb_test, model.predict(Xb_test))
    label = MODEL_LABELS[key]
    compare[label] = te
    print(f"{label:32s} train={tr:.4f}  test={te:.4f}")

best = max(compare, key=compare.get)
print(f"\nテスト正解率が最も高いモデル: {best} ({compare[best]:.4f})")


In [ ]:
# === ✍️ 問題3 解答（採点対象）===

# (3-a) 設計判断2-1：ノイズ・外れ値が多く過学習を避けたいとき → 選んだ手法（A〜F）：(　)
# (A) 単一決定木
# (B) RandomForest（バギング）
# (C) GradientBoosting（ブースティング）
# (D) XGBoost（ブースティング系）
# (E) LogisticRegression（線形・アンサンブルなし）
# (F) スタッキング
design2_1_choice = ""

# (3-b) その理由
answer_3_b = """
"""

# (3-c) 設計判断2-2：チューニングしてでも最高精度を出したいとき → 選んだ手法（A〜F）：(　)
# (A) 単一決定木
# (B) RandomForest（バギング）
# (C) GradientBoosting（ブースティング）
# (D) XGBoost（ブースティング系）
# (E) LogisticRegression（線形・アンサンブルなし）
# (F) スタッキング
design2_2_choice = ""

# (3-d) その理由
answer_3_d = """
"""

# (3-e) 考察3：シンプルなデータでも実務で XGBoost が好まれる理由を1つ
reflection3 = """
"""



## 問題4　特徴量重要度のコードを説明する　【説明】
***

### 特徴量重要度とは

ランダムフォレストの各木では，特徴量を使って分岐するたびに不純度（ジニ係数やエントロピー）が減少する。`feature_importances_` は各特徴量が**全ての木での不純度減少量の合計**に占める割合を示す。合計は1になるよう正規化されている。

### グラフを重要度の降順に並べる方法

```python
indices = np.argsort(importances)[::-1]  # 降順にソート
plt.bar(range(len(names)), importances[indices])
```

降順に並べると「どの特徴量が最も影響力があるか」が一目でわかる。

### 注意：特徴量重要度の落とし穴

1. **相関特徴量がある場合**: `Age` と `Fare` が相関していると，どちらか一方の重要度が不当に低くなる
2. **カテゴリ変数のバイアス**: 取り得る値の数が多い変数ほど重要度が高く見えやすい
3. **因果関係ではない**: 「Pclass が重要」は「客室クラスが生存を決める原因だ」ではなく，「客室クラスが生存と強く関連している」という意味だ

> **卒業研究での使い方**: 特徴量重要度は「どの変数を調査すべきか」の優先度付けに使えるが，解釈は慎重に行うこと。SHAP (SHapley Additive exPlanations) というより高度な解釈手法も存在する（**発展**）。

### 課題

問題1で学習した RandomForest（`rf`）の `feature_importances_` を取得し、**重要度の降順**に並べて棒グラフにするコードを用意しました。

下のコードセルは **完成形**です。**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたらセルを実行し、グラフと重要度のランキングを確認してください。

説明を書くときは、次の問いを意識してください：

- `feature_importances_` には何が入っていて、合計はいくつになるか？
- `np.argsort(importances)[::-1]` は何をして「降順の並び」を作っているか？
- グラフはなぜ降順に並べると読み取りやすいのか？

> **設計判断（重要だった特徴量）**: 出力されたランキングを見て、**最も重要だった特徴量を上位2〜3個**挙げ、それが生存予測に効くのは「なぜ妥当か」を解答用コードセルに書いてください。
>
> **考察4（落とし穴）**: `Age` と `Fare` のように**相関する特徴量**があると、重要度はどう歪む可能性がありますか？ また「重要度が高い＝その変数が生存の原因」と言ってよいですか？（`この回で学ぶこと` の注意点を参照）

In [ ]:
# 各行の「# 説明:」に自分の言葉で意味を書いてください（コードは変更しない）。
# （説明は AI に書かせず、自分で書くこと。rf と X は問題1で作成済み）

importances = rf.feature_importances_       # 説明:（何の値が入っている？合計はいくつ？）
feature_names = X.columns                   # 説明:（問題1の説明変数の名前）

indices = np.argsort(importances)[::-1]     # 説明:（argsort と [::-1] で何をしている？）

plt.figure(figsize=(8, 4))
plt.bar(range(len(feature_names)), importances[indices])                       # 説明:（降順の重要度を棒グラフに）
plt.xticks(range(len(feature_names)), feature_names[indices], rotation=45, ha="right")  # 説明:
plt.ylabel("重要度")
plt.title("RandomForest の特徴量重要度（降順）")
plt.tight_layout()
plt.show()

print("重要度ランキング:")
for rank, i in enumerate(indices, start=1):
    print(f"{rank}. {feature_names[i]:10s}: {importances[i]:.4f}")   # 説明:


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (4-a) `feature_importances_` には何が入っていて、合計はいくつになるか
answer_4_a = """
"""

# (4-b) 設計判断：最も重要だった特徴量（上位2〜3個）
answer_4_b = """
"""

# (4-c) それが生存予測に効くのはなぜ妥当か
answer_4_c = """
"""

# (4-d) 考察4：相関する特徴量があると重要度はどう歪むか／「重要度が高い＝原因」と言えるか
reflection4 = """
"""

